
# Pertemuan 3


In [6]:

import pandas as pd
import numpy as np
import requests
from scipy.stats import zscore


In [13]:
url = 'https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'

df = pd.read_csv(url)

print("Dataset berhasil dibaca")
print(df.head())

Dataset berhasil dibaca
   id  luas_m2  harga_juta   kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  jogja    2.0          2000    baik
1   2    254.0       761.0  Medan    NaN          1995   Bagus
2   3    249.7       895.0  Depok    NaN          1983    baik
3   4     49.7       178.0    YGY    5.0          2013    baik
4   5    133.4       424.0  Medan    5.0          2004  Sedang


In [14]:

# Eksplorasi awal dataset
print("=== INFO DATASET ===")
df.info()

print("\n=== 5 DATA TERATAS ===")
print(df.head())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())


=== INFO DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

=== 5 DATA TERATAS ===
   id  luas_m2  harga_juta   kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  jogja    2.0          2000    baik
1   2    254.0       761.0  Medan    NaN          1995   Bagus
2   3    249.7       895.0  Depok    NaN          1983    baik
3   4     49.7       178.0    YGY    5.0          2013    baik
4   5    133.4       424.0  Medan    5.0          2004  Sedang

=== MISSING VALUES ===


In [15]:

# Mengecek data duplikat
jumlah_duplikat = df.duplicated().sum()
print("Jumlah data duplikat:", jumlah_duplikat)

# Menghapus duplikat
df = df.drop_duplicates()

print("Shape setelah hapus duplikat:", df.shape)


Jumlah data duplikat: 0
Shape setelah hapus duplikat: (130, 7)


In [16]:

# Normalisasi teks/string
df['kota'] = df['kota'].str.strip().str.capitalize()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print("Normalisasi string selesai")
print(df[['kota','kondisi']].head())


Normalisasi string selesai
    kota kondisi
0  Jogja    baik
1  Medan   bagus
2  Depok    baik
3    Ygy    baik
4  Medan  sedang


In [17]:

# Imputasi missing values

# Numerik → median
for kolom in ['luas_m2', 'harga_juta', 'kamar']:
    df[kolom] = df[kolom].fillna(df[kolom].median())

# Kategorik → modus
for kolom in ['kota', 'kondisi']:
    df[kolom] = df[kolom].fillna(df[kolom].mode()[0])

print("Missing values berhasil ditangani")
print(df.isnull().sum())


Missing values berhasil ditangani
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


In [18]:

# Menangani nilai tidak valid
# Tahun bangun di bawah 1950 dianggap tidak valid

df.loc[df['tahun_bangun'] < 1950, 'tahun_bangun'] = df['tahun_bangun'].median()

print("Nilai tidak valid berhasil diperbaiki")


Nilai tidak valid berhasil diperbaiki


In [19]:

# Deteksi dan penanganan outlier dengan metode IQR

def iqr_capping(data, kolom):
    q1 = data[kolom].quantile(0.25)
    q3 = data[kolom].quantile(0.75)
    iqr = q3 - q1

    batas_bawah = q1 - (1.5 * iqr)
    batas_atas = q3 + (1.5 * iqr)

    data[kolom] = data[kolom].clip(batas_bawah, batas_atas)

    return data

for kolom in ['luas_m2', 'harga_juta']:
    df = iqr_capping(df, kolom)

print("Outlier berhasil ditangani")


Outlier berhasil ditangani


In [20]:

# Validasi akhir
print("=== VALIDASI ===")
print("Total missing value:", df.isnull().sum().sum())
print("Total data duplikat:", df.duplicated().sum())

print("\nDeskripsi statistik setelah cleaning")
print(df.describe())


=== VALIDASI ===
Total missing value: 0
Total data duplikat: 0

Deskripsi statistik setelah cleaning
               id     luas_m2   harga_juta       kamar  tahun_bangun
count  130.000000  130.000000   130.000000  130.000000    130.000000
mean    65.500000  188.274423   686.490385    3.476923   2063.500000
std     37.671829   95.297150   404.633957    1.712767    701.539174
min      1.000000  -50.000000  -422.750000    1.000000   1980.000000
25%     33.250000  101.600000   380.500000    2.000000   1992.250000
50%     65.500000  193.800000   655.000000    4.000000   2002.000000
75%     97.750000  266.150000   916.000000    5.000000   2011.750000
max    130.000000  512.975000  1719.250000    6.000000   9999.000000


In [21]:

# Export dataset bersih
df.to_csv('housing_clean_final.csv', index=False)

print("Dataset bersih berhasil disimpan")


Dataset bersih berhasil disimpan


In [22]:

# Mengakses REST API

url = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(url, timeout=10)

    if response.status_code == 200:
        data = response.json()

        df_api = pd.json_normalize(data)

        print("API berhasil diakses")
        print(df_api[['id', 'name', 'email', 'address.city']].head())

    else:
        print("Terjadi error:", response.status_code)

except Exception as e:
    print("Gagal koneksi ke API")
    print(e)


API berhasil diakses
   id              name                      email   address.city
0   1     Leanne Graham          Sincere@april.biz    Gwenborough
1   2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh
2   3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven
3   4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview
